# Next Priority Runs — Exp43 / VS r32

Outputs are Drive-only. This notebook does not update `PROGRESS.md` and does not push result CSVs to GitHub.

Preset order:
- `p0_exp43_s24`: Exp43 C0/C1 on S24, VS init r=32
- `p1_exp43_s01`: Exp43 C0/C1 on S01, VS init r=32
- `p2_vs_s28`: S28 r=32 SD LoRA VS generation
- `p3_vs_s02`: S02 r=32 SD LoRA VS generation


In [1]:
# [1] GPU check
import torch
assert torch.cuda.is_available(), 'GPU 없음: Runtime > Change runtime type > GPU'
print('torch', torch.__version__)
print('GPU  ', torch.cuda.get_device_name(0))


torch 2.11.0+cu128
GPU   NVIDIA L4


In [2]:
# [2] Mount Drive and clone/pull repo
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess
REPO = 'https://github.com/heegyukim4043/test_diffusion_EEG_visualstim_image.git'
WORKDIR = '/content/vsvi_project'

if os.path.exists(WORKDIR) and os.path.exists(os.path.join(WORKDIR, '.git')):
    subprocess.run(['git', '-C', WORKDIR, 'pull', 'origin', 'main'], check=True)
else:
    subprocess.run(['rm', '-rf', WORKDIR], check=True)
    subprocess.run(['git', 'clone', REPO, WORKDIR], check=True)

os.chdir(WORKDIR)
subprocess.run(['git', 'log', '--oneline', '-5'], check=False)


Mounted at /content/drive


CompletedProcess(args=['git', 'log', '--oneline', '-5'], returncode=0)

In [ ]:
# [3] Select preset and launch background run
# Change PRESET one at a time. Do not run multiple presets simultaneously on one GPU.
PRESET = 'p1_exp43_s01' # 'p0_exp43_s24'  # p0_exp43_s24 | p1_exp43_s01 | p2_vs_s28 | p3_vs_s02

# Exp43 default uses VI/class=60 => 432 train trials. Set FULL_VI=True for all VI trials.
FULL_VI = False

cmd = f"python -u launch_next_priority_runs.py --preset {PRESET}"
if FULL_VI:
    cmd += ' --full_vi'

print(cmd)
!cd /content/vsvi_project && {cmd}


In [ ]:
# [4] Monitor active process and GPU
!pgrep -af 'train_exp43_vi_lora.py|train_vs_re_lora_gen.py' || true
!nvidia-smi
!ls -lt /content/drive/MyDrive/vsvi_data/logs | head -20


In [ ]:
# [5] Tail latest log for the selected preset
import glob, os
patterns = {
    'p0_exp43_s24': '/content/drive/MyDrive/vsvi_data/logs/exp43_s24_c0c1_r32_tok16_*.log',
    'p1_exp43_s01': '/content/drive/MyDrive/vsvi_data/logs/exp43_s01_c0c1_r32_tok16_*.log',
    'p2_vs_s28': '/content/drive/MyDrive/vsvi_data/logs/vs_s28_lora_r32_tok16_*.log',
    'p3_vs_s02': '/content/drive/MyDrive/vsvi_data/logs/vs_s02_lora_r32_tok16_*.log',
}
logs = sorted(glob.glob(patterns[PRESET]), key=os.path.getmtime, reverse=True)
assert logs, f'No log found for {PRESET}'
LOG = logs[0]
print('LOG=', LOG)
!tail -n 120 "{LOG}"


In [ ]:
# [6] Find Drive-only result CSVs
!echo '=== Exp43 VI results ==='
!find /content/drive/MyDrive/vsvi_data/checkpoints_exp43_vi_lora -name 'results_exp43_vi_lora.csv' -ls 2>/dev/null | tail -30
!echo '=== VS results ==='
!find /content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen -name 'results_lora_gen.csv' -ls 2>/dev/null | tail -30


In [ ]:
import os, shutil
if os.path.exists('/content/drive'):
    shutil.rmtree('/content/drive')
from google.colab import drive
drive.mount('/content/drive')

OSError: [Errno 125] Operation canceled: '/content/drive/.Encrypted/.shortcut-targets-by-id'

In [ ]:
%%bash
ls /content/drive/MyDrive/vsvi_data/preproc_vi_re/preproc_subj_01_1.npz && echo "✅ Drive OK" || echo "❌ Drive 문제"

/content/drive/MyDrive/vsvi_data/preproc_vi_re/preproc_subj_01_1.npz
✅ Drive OK


In [ ]:
%%bash
# 환경 복구
cd /content
git clone https://github.com/heegyukim4043/test_diffusion_EEG_visualstim_image.git vsvi_project 2>/dev/null
cd /content/vsvi_project && git pull
ln -sfn /content/drive/MyDrive/vsvi_data/preproc_vi_re /content/vsvi_project/preproc_vi_re
ln -sfn /content/drive/MyDrive/vsvi_data/checkpoints_vsre_dino /content/vsvi_project/checkpoints_vsre_dino
pip uninstall -y torchao 2>/dev/null

ls /content/vsvi_project/preproc_vi_re/preproc_subj_02_1.npz && echo "✅ Data OK" || exit 1

# S02 Exp43
python -u train_exp43_vi_lora.py \
  --subject_ids 2 \
  --conditions c0,c1 \
  --data_root /content/vsvi_project/preproc_vi_re \
  --img_root /content/vsvi_project/preproc_data_vi/images \
  --supcon_ckpt /content/drive/MyDrive/vsvi_data/checkpoints_vsre_dino/20260707_130522_ch32_merged_ep200_supcon \
  --init_lora_ckpt /content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260707_133400_lora_r32_ep100/subj02_lora_best.pt \
  --ckpt_root /content/drive/MyDrive/vsvi_data/checkpoints_exp43_vi_lora \
  --lora_r 32 \
  --n_eeg_tokens 16 \
  --epochs 100 \
  --batch_size 2 \
  --per_class_total 135 \
  --eval_n_samples 54 \
  --fp16 \
  2>&1 | tee /content/drive/MyDrive/vsvi_data/logs/exp43_s02_c0c1_r32_tok16_full_$(date +%Y%m%d_%H%M%S).log

In [ ]:
%%bash
cd /content/vsvi_project
rm -rf /content/vsvi_project/preproc_vs_re
ln -sfn /content/drive/MyDrive/vsvi_data/preproc_vs_re /content/vsvi_project/preproc_vs_re

python -u train_vs_re_dino.py \
  --subject_ids 28 \
  --data_root /content/vsvi_project/preproc_vs_re \
  --img_root /content/vsvi_project/preproc_data_vi/images \
  --ckpt_root /content/drive/MyDrive/vsvi_data/checkpoints_vsre_dino \
  --loss_type supcon \
  --encoder_type v2 \
  --epochs 200 \
  --batch_size 4 \
  2>&1 | tee /content/drive/MyDrive/vsvi_data/logs/supcon_s28_$(date +%Y%m%d_%H%M%S).log

In [ ]:
%%bash
cd /content/vsvi_project

python -u train_vs_re_dino.py \
  --subject_ids 29 \
  --data_root /content/vsvi_project/preproc_vs_re \
  --img_root /content/vsvi_project/preproc_data_vi/images \
  --ckpt_root /content/drive/MyDrive/vsvi_data/checkpoints_vsre_dino \
  --loss_type supcon \
  --encoder_type v2 \
  --epochs 200 \
  --batch_size 4 \
  2>&1 | tee /content/drive/MyDrive/vsvi_data/logs/supcon_s29_$(date +%Y%m%d_%H%M%S).log

In [ ]:
import os, shutil
# 잔여 제거
try:
    if os.path.exists('/content/drive'):
        shutil.rmtree('/content/drive')
except:
    pass

from google.colab import drive
drive.mount('/content/drive')
print("Drive:", os.path.exists('/content/drive/MyDrive/vsvi_data/preproc_vs_re'))

In [ ]:
import os
# 마운트 상태 확인
print("MyDrive:", os.path.exists('/content/drive/MyDrive'))
print("내용:", os.listdir('/content/drive/MyDrive')[:10] if os.path.exists('/content/drive/MyDrive') else "없음")

MyDrive: True
내용: ['Quince · SlidesCarnival의 사본.gslides', 'Hyperscanning_', 'DSI 24.pptx', 'EEG&연구설명 발표자료.pptx', 'Print', 'vsvi_data', 'MI_loso_project', 'project.tar.gz']


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
import os
print("vsvi_data:", os.path.exists('/content/drive/MyDrive/vsvi_data'))

Mounted at /content/drive
vsvi_data: True


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os
print("vsvi_data:", os.path.exists('/content/drive/MyDrive/vsvi_data/preproc_vi_re'))

In [ ]:
import os
if not os.path.exists('/content/drive/MyDrive/vsvi_data'):
    from google.colab import drive
    drive.mount('/content/drive')
print("Drive:", os.path.exists('/content/drive/MyDrive/vsvi_data/preproc_vs_re'))

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os
print("vsvi_data:", os.path.exists('/content/drive/MyDrive/vsvi_data/preproc_vs_re'))
print("S09 파일:", os.path.exists('/content/drive/MyDrive/vsvi_data/preproc_vs_re/preproc_subj_09_1.npz'))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
print("S09 vs:", os.path.exists('/content/drive/MyDrive/vsvi_data/preproc_vs_re/preproc_subj_09_1.npz'))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
S09 vs: True


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
print("vi_re:", os.path.exists('/content/drive/MyDrive/vsvi_data/preproc_vi_re/preproc_subj_09_1.npz'))
print("vs_re:", os.path.exists('/content/drive/MyDrive/vsvi_data/preproc_vs_re/preproc_subj_09_1.npz'))

Mounted at /content/drive
vi_re: True
vs_re: True


In [ ]:
%%bash
cd /content/vsvi_project

# 실제 디렉터리 제거 (rm -rf)
rm -rf /content/vsvi_project/preproc_vs_re
rm -rf /content/vsvi_project/preproc_vi_re

# symlink 생성
ln -s /content/drive/MyDrive/vsvi_data/preproc_vs_re /content/vsvi_project/preproc_vs_re
ln -s /content/drive/MyDrive/vsvi_data/preproc_vi_re /content/vsvi_project/preproc_vi_re

# 확인
ls -la preproc_vs_re/preproc_subj_09_1.npz && echo "✅ OK" || echo "❌ 실패"

-rw------- 1 root root 16132799 Jul  9 18:05 preproc_vs_re/preproc_subj_09_1.npz
✅ OK


In [ ]:
%%bash
cd /content/vsvi_project 2>/dev/null || (cd /content && git clone https://github.com/heegyukim4043/test_diffusion_EEG_visualstim_image.git vsvi_project && cd vsvi_project)
cd /content/vsvi_project
rm -f preproc_vs_re
ln -sfn /content/drive/MyDrive/vsvi_data/preproc_vs_re /content/vsvi_project/preproc_vs_re

ls preproc_vs_re/preproc_subj_09_1.npz && echo "✅ OK" || exit 1

python -u train_vs_re_dino.py \
  --subject_ids 9 \
  --data_root /content/vsvi_project/preproc_vs_re \
  --img_root /content/vsvi_project/preproc_data_vi/images \
  --ckpt_root /content/drive/MyDrive/vsvi_data/checkpoints_vsre_dino \
  --loss_type supcon \
  --encoder_type v2 \
  --epochs 200 \
  --batch_size 4 \
  2>&1 | tee /content/drive/MyDrive/vsvi_data/logs/supcon_s09_$(date +%Y%m%d_%H%M%S).log

preproc_vs_re/preproc_subj_09_1.npz
✅ OK
[INFO] Device: cuda  n_ch=32  max_sessions=None  loss_type=supcon
[INFO] Subjects (1): [9]
[INFO] Session counts: {9: 4}
[INFO] Loading DINO: dinov2_vits14
Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vi

In [ ]:
%%bash
cd /content/vsvi_project

# S09 데이터에 NaN이나 이상값이 있는지 확인
python3 -c "
import numpy as np
import glob

for sid in [9, 28, 29]:
    files = sorted(glob.glob(f'/content/vsvi_project/preproc_vs_re/preproc_subj_{sid:02d}_*.npz'))
    print(f'=== S{sid:02d} ({len(files)} sessions) ===')
    for f in files[:2]:
        d = np.load(f)
        keys = list(d.keys())
        X = d[keys[0]]
        print(f'  {f.split(\"/\")[-1]}: shape={X.shape}, NaN={np.isnan(X).any()}, min={X.min():.3f}, max={X.max():.3f}, mean={X.mean():.3f}')
"

=== S09 (4 sessions) ===
  preproc_subj_09_1.npz: shape=(135, 32, 2048), NaN=False, min=-242.250, max=397.500, mean=-0.609
  preproc_subj_09_2.npz: shape=(135, 32, 2048), NaN=False, min=-545.000, max=596.500, mean=-0.358
=== S28 (6 sessions) ===
  preproc_subj_28_1.npz: shape=(135, 32, 2048), NaN=False, min=-inf, max=inf, mean=nan
  preproc_subj_28_2.npz: shape=(135, 32, 2048), NaN=False, min=-158.625, max=285.750, mean=-0.360
=== S29 (5 sessions) ===
  preproc_subj_29_1.npz: shape=(135, 32, 2048), NaN=False, min=-219.750, max=360.000, mean=-0.312
  preproc_subj_29_2.npz: shape=(135, 32, 2048), NaN=False, min=-276.250, max=380.250, mean=-0.338


/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:127: RuntimeWarning: invalid value encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)


In [ ]:
%%bash
cd /content/vsvi_project
python3 -c "
import numpy as np, glob
files = sorted(glob.glob('/content/vsvi_project/preproc_vs_re/preproc_subj_28_*.npz'))
for f in files:
    d = np.load(f)
    X = d[list(d.keys())[0]]
    n_inf = np.isinf(X).sum()
    print(f'{f.split(\"/\")[-1]}: inf 개수={n_inf}, shape={X.shape}')
"

preproc_subj_28_1.npz: inf 개수=44767, shape=(135, 32, 2048)
preproc_subj_28_2.npz: inf 개수=0, shape=(135, 32, 2048)
preproc_subj_28_3.npz: inf 개수=0, shape=(135, 32, 2048)
preproc_subj_28_4.npz: inf 개수=0, shape=(135, 32, 2048)
preproc_subj_28_5.npz: inf 개수=0, shape=(108, 32, 2048)
preproc_subj_28_6.npz: inf 개수=0, shape=(135, 32, 2048)


In [ ]:
%%bash
cd /content/vsvi_project

python -u train_vs_re_dino.py \
  --subject_ids 9 \
  --data_root /content/vsvi_project/preproc_vs_re \
  --img_root /content/vsvi_project/preproc_data_vi/images \
  --ckpt_root /content/drive/MyDrive/vsvi_data/checkpoints_vsre_dino \
  --loss_type supcon \
  --encoder_type v2 \
  --epochs 200 \
  --batch_size 4 \
  --lr 1e-4 \
  --temperature 0.07 \
  2>&1 | tee /content/drive/MyDrive/vsvi_data/logs/supcon_s09_retry_$(date +%Y%m%d_%H%M%S).log

[INFO] Device: cuda  n_ch=32  max_sessions=None  loss_type=supcon
[INFO] Subjects (1): [9]
[INFO] Session counts: {9: 4}
[INFO] Loading DINO: dinov2_vits14
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
  proto_dino: torch.Size([9, 384])

  Subject 09  (sessions=4)
  [dataset] S09 sess1: npz  shape=(135, 32, 2048)
  [dataset] S09 sess2: npz  shape=(135, 32, 2048)
  [dataset] S09 sess3: npz  shape=(135, 32, 2048)
  [dataset] S09 sess4: npz  shape=(108, 32, 2048)
  [dataset] S09  trials=5

In [ ]:
%%bash
# S28 sess1을 백업 (로더가 안 읽도록 확장자 변경)
mv /content/drive/MyDrive/vsvi_data/preproc_vs_re/preproc_subj_28_1.npz \
   /content/drive/MyDrive/vsvi_data/preproc_vs_re/preproc_subj_28_1.npz.bad

# 확인
ls /content/drive/MyDrive/vsvi_data/preproc_vs_re/preproc_subj_28_*.npz

/content/drive/MyDrive/vsvi_data/preproc_vs_re/preproc_subj_28_2.npz
/content/drive/MyDrive/vsvi_data/preproc_vs_re/preproc_subj_28_3.npz
/content/drive/MyDrive/vsvi_data/preproc_vs_re/preproc_subj_28_4.npz
/content/drive/MyDrive/vsvi_data/preproc_vs_re/preproc_subj_28_5.npz
/content/drive/MyDrive/vsvi_data/preproc_vs_re/preproc_subj_28_6.npz


In [ ]:
%%bash
cd /content/vsvi_project
rm -rf preproc_vs_re
ln -s /content/drive/MyDrive/vsvi_data/preproc_vs_re /content/vsvi_project/preproc_vs_re

python -u train_vs_re_dino.py \
  --subject_ids 28 \
  --data_root /content/vsvi_project/preproc_vs_re \
  --img_root /content/vsvi_project/preproc_data_vi/images \
  --ckpt_root /content/drive/MyDrive/vsvi_data/checkpoints_vsre_dino \
  --loss_type supcon \
  --encoder_type v2 \
  --epochs 200 \
  --batch_size 4 \
  --lr 1e-4 \
  --temperature 0.07 \
  2>&1 | tee /content/drive/MyDrive/vsvi_data/logs/supcon_s28_retry_$(date +%Y%m%d_%H%M%S).log

[INFO] Device: cuda  n_ch=32  max_sessions=None  loss_type=supcon
[INFO] Subjects (1): [28]
[INFO] Session counts: {28: 5}
[INFO] Loading DINO: dinov2_vits14
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
  proto_dino: torch.Size([9, 384])

  Subject 28  (sessions=5)
  [dataset] S28 sess2: npz  shape=(135, 32, 2048)
  [dataset] S28 sess3: npz  shape=(135, 32, 2048)
  [dataset] S28 sess4: npz  shape=(135, 32, 2048)
  [dataset] S28 sess5: npz  shape=(108, 32, 2048)
  [dataset] S28 sess6: 

In [ ]:
%%bash
cd /content/vsvi_project

python -u train_vs_re_dino.py \
  --subject_ids 29 \
  --data_root /content/vsvi_project/preproc_vs_re \
  --img_root /content/vsvi_project/preproc_data_vi/images \
  --ckpt_root /content/drive/MyDrive/vsvi_data/checkpoints_vsre_dino \
  --loss_type supcon \
  --encoder_type v2 \
  --epochs 200 \
  --batch_size 4 \
  --lr 1e-4 \
  --temperature 0.07 \
  2>&1 | tee /content/drive/MyDrive/vsvi_data/logs/supcon_s29_retry_$(date +%Y%m%d_%H%M%S).log

[INFO] Device: cuda  n_ch=32  max_sessions=None  loss_type=supcon
[INFO] Subjects (1): [29]
[INFO] Session counts: {29: 5}
[INFO] Loading DINO: dinov2_vits14
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
  proto_dino: torch.Size([9, 384])

  Subject 29  (sessions=5)
  [dataset] S29 sess1: npz  shape=(135, 32, 2048)
  [dataset] S29 sess2: npz  shape=(135, 32, 2048)
  [dataset] S29 sess3: npz  shape=(135, 32, 2048)
  [dataset] S29 sess4: npz  shape=(135, 32, 2048)
  [dataset] S29 sess5: 

In [ ]:
%%bash
pip uninstall -y torchao 2>/dev/null

cd /content/vsvi_project

python -u train_vs_re_lora_gen.py \
  --subject_ids 28 \
  --data_root /content/vsvi_project/preproc_vs_re \
  --img_root /content/vsvi_project/preproc_data_vi/images \
  --supcon_ckpt /content/drive/MyDrive/vsvi_data/checkpoints_vsre_dino/20260709_190207_ch32_merged_ep200_supcon \
  --ckpt_root /content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen \
  --lora_r 32 \
  --n_eeg_tokens 16 \
  --epochs 100 \
  --batch_size 2 \
  --fp16 \
  2>&1 | tee /content/drive/MyDrive/vsvi_data/logs/vs_lora_s28_r32_$(date +%Y%m%d_%H%M%S).log

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
[INFO] Device: cuda  SD1.5 LoRA generator
[INFO] Loading DINO...
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
[INFO] Loading SD VAE...
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migratin

In [ ]:
%%bash
pip uninstall -y torchao 2>/dev/null

cd /content/vsvi_project
rm -rf preproc_vi_re
ln -s /content/drive/MyDrive/vsvi_data/preproc_vi_re /content/vsvi_project/preproc_vi_re

python -u train_exp43_vi_lora.py \
  --subject_ids 28 \
  --conditions c0,c1 \
  --data_root /content/vsvi_project/preproc_vi_re \
  --img_root /content/vsvi_project/preproc_data_vi/images \
  --supcon_ckpt /content/drive/MyDrive/vsvi_data/checkpoints_vsre_dino/20260709_190207_ch32_merged_ep200_supcon \
  --init_lora_ckpt /content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260709_191616_lora_r32_ep100/subj28_lora_best.pt \
  --ckpt_root /content/drive/MyDrive/vsvi_data/checkpoints_exp43_vi_lora \
  --lora_r 32 \
  --n_eeg_tokens 16 \
  --epochs 100 \
  --batch_size 2 \
  --per_class_total 90 \
  --eval_n_samples 54 \
  --fp16 \
  2>&1 | tee /content/drive/MyDrive/vsvi_data/logs/exp43_s28_c0c1_r32_tok16_$(date +%Y%m%d_%H%M%S).log

[INFO] Device: cuda  Exp43 VI LoRA
[INFO] Loading DINO...
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
[INFO] Loading SD VAE...
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_

In [ ]:
%%bash
cd /content/vsvi_project
python3 -c "
import numpy as np, glob
files = sorted(glob.glob('/content/vsvi_project/preproc_vi_re/preproc_subj_28_*.npz'))
for f in files:
    d = np.load(f)
    X = d[list(d.keys())[0]]
    n_inf = np.isinf(X).sum()
    n_nan = np.isnan(X).sum()
    print(f'{f.split(\"/\")[-1]}: inf={n_inf}, nan={n_nan}, shape={X.shape}')
"

bash: line 1: cd: /content/vsvi_project: No such file or directory


In [ ]:
%%bash
cd /content/vsvi_project
python3 -c "
import numpy as np, glob
for sid in [28, 29, 9]:
    files = sorted(glob.glob(f'/content/vsvi_project/preproc_vi_re/preproc_subj_{sid:02d}_*.npz'))
    print(f'=== S{sid:02d} VI ===')
    for f in files:
        d = np.load(f)
        X = d[list(d.keys())[0]]
        print(f'  {f.split(\"/\")[-1]}: inf={np.isinf(X).sum()}, nan={np.isnan(X).sum()}')
"

=== S28 VI ===
  preproc_subj_28_1.npz: inf=0, nan=0
  preproc_subj_28_2.npz: inf=0, nan=0
  preproc_subj_28_3.npz: inf=0, nan=0
  preproc_subj_28_4.npz: inf=0, nan=0
  preproc_subj_28_5.npz: inf=33673, nan=0
  preproc_subj_28_6.npz: inf=0, nan=0
=== S29 VI ===
  preproc_subj_29_1.npz: inf=0, nan=0
  preproc_subj_29_2.npz: inf=0, nan=0
  preproc_subj_29_3.npz: inf=0, nan=0
  preproc_subj_29_4.npz: inf=0, nan=0
  preproc_subj_29_5.npz: inf=0, nan=0
=== S09 VI ===
  preproc_subj_09_1.npz: inf=0, nan=0
  preproc_subj_09_2.npz: inf=0, nan=0
  preproc_subj_09_3.npz: inf=0, nan=0
  preproc_subj_09_4.npz: inf=0, nan=0


In [ ]:
%%bash
# S28 VI sess5 제외
mv /content/drive/MyDrive/vsvi_data/preproc_vi_re/preproc_subj_28_5.npz \
   /content/drive/MyDrive/vsvi_data/preproc_vi_re/preproc_subj_28_5.npz.bad

# 확인
ls /content/drive/MyDrive/vsvi_data/preproc_vi_re/preproc_subj_28_*.npz

/content/drive/MyDrive/vsvi_data/preproc_vi_re/preproc_subj_28_1.npz
/content/drive/MyDrive/vsvi_data/preproc_vi_re/preproc_subj_28_2.npz
/content/drive/MyDrive/vsvi_data/preproc_vi_re/preproc_subj_28_3.npz
/content/drive/MyDrive/vsvi_data/preproc_vi_re/preproc_subj_28_4.npz
/content/drive/MyDrive/vsvi_data/preproc_vi_re/preproc_subj_28_6.npz


mv: cannot stat '/content/drive/MyDrive/vsvi_data/preproc_vi_re/preproc_subj_28_5.npz': No such file or directory


In [ ]:
%%bash
pip uninstall -y torchao 2>/dev/null

cd /content/vsvi_project
rm -rf preproc_vi_re
ln -s /content/drive/MyDrive/vsvi_data/preproc_vi_re /content/vsvi_project/preproc_vi_re

python -u train_exp43_vi_lora.py \
  --subject_ids 28 \
  --conditions c0,c1 \
  --data_root /content/vsvi_project/preproc_vi_re \
  --img_root /content/vsvi_project/preproc_data_vi/images \
  --supcon_ckpt /content/drive/MyDrive/vsvi_data/checkpoints_vsre_dino/20260709_190207_ch32_merged_ep200_supcon \
  --init_lora_ckpt /content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260709_191616_lora_r32_ep100/subj28_lora_best.pt \
  --ckpt_root /content/drive/MyDrive/vsvi_data/checkpoints_exp43_vi_lora \
  --lora_r 32 \
  --n_eeg_tokens 16 \
  --epochs 100 \
  --batch_size 2 \
  --per_class_total 75 \
  --eval_n_samples 45 \
  --fp16 \
  2>&1 | tee /content/drive/MyDrive/vsvi_data/logs/exp43_s28_c0c1_r32_tok16_$(date +%Y%m%d_%H%M%S).log

[INFO] Device: cuda  Exp43 VI LoRA
[INFO] Loading DINO...
[INFO] local DINO cache not found, downloading facebookresearch/dinov2...
Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth
100%|██████████| 84.2M/84.2M [00:00<00:00, 228M

In [ ]:
%%bash
pip uninstall -y torchao 2>/dev/null
cd /content/vsvi_project
rm -rf preproc_vs_re
ln -s /content/drive/MyDrive/vsvi_data/preproc_vs_re /content/vsvi_project/preproc_vs_re

python -u train_vs_re_lora_gen.py \
  --subject_ids 29 \
  --data_root /content/vsvi_project/preproc_vs_re \
  --img_root /content/vsvi_project/preproc_data_vi/images \
  --supcon_ckpt /content/drive/MyDrive/vsvi_data/checkpoints_vsre_dino/20260709_191010_ch32_merged_ep200_supcon \
  --ckpt_root /content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen \
  --lora_r 32 \
  --n_eeg_tokens 16 \
  --epochs 100 \
  --batch_size 2 \
  --fp16 \
  2>&1 | tee /content/drive/MyDrive/vsvi_data/logs/vs_lora_s29_r32_$(date +%Y%m%d_%H%M%S).log

[INFO] Device: cuda  SD1.5 LoRA generator
[INFO] Loading DINO...
[INFO]  local cache not found, downloading from facebookresearch/dinov2...
Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth
100%|██████████| 84.2M/84.2M [00:00<00:

In [ ]:
%%bash
pip uninstall -y torchao 2>/dev/null

cd /content/vsvi_project
rm -rf preproc_vi_re
ln -s /content/drive/MyDrive/vsvi_data/preproc_vi_re /content/vsvi_project/preproc_vi_re

python -u train_exp43_vi_lora.py \
  --subject_ids 29 \
  --conditions c0,c1 \
  --data_root /content/vsvi_project/preproc_vi_re \
  --img_root /content/vsvi_project/preproc_data_vi/images \
  --supcon_ckpt /content/drive/MyDrive/vsvi_data/checkpoints_vsre_dino/20260709_191010_ch32_merged_ep200_supcon \
  --init_lora_ckpt /content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260710_003529_lora_r32_ep100/subj29_lora_best.pt \
  --ckpt_root /content/drive/MyDrive/vsvi_data/checkpoints_exp43_vi_lora \
  --lora_r 32 \
  --n_eeg_tokens 16 \
  --epochs 100 \
  --batch_size 2 \
  --per_class_total 75 \
  --eval_n_samples 45 \
  --fp16 \
  2>&1 | tee /content/drive/MyDrive/vsvi_data/logs/exp43_s29_c0c1_r32_tok16_$(date +%Y%m%d_%H%M%S).log

Process is terminated.


In [ ]:
%%bash
pip uninstall -y torchao 2>/dev/null
cd /content/vsvi_project
rm -rf preproc_vs_re
ln -s /content/drive/MyDrive/vsvi_data/preproc_vs_re /content/vsvi_project/preproc_vs_re

python -u train_vs_re_lora_gen.py \
  --subject_ids 9 \
  --data_root /content/vsvi_project/preproc_vs_re \
  --img_root /content/vsvi_project/preproc_data_vi/images \
  --supcon_ckpt /content/drive/MyDrive/vsvi_data/checkpoints_vsre_dino/20260709_185832_ch32_merged_ep200_supcon \
  --ckpt_root /content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen \
  --lora_r 32 \
  --n_eeg_tokens 16 \
  --epochs 100 \
  --batch_size 2 \
  --fp16 \
  2>&1 | tee /content/drive/MyDrive/vsvi_data/logs/vs_lora_s09_r32_$(date +%Y%m%d_%H%M%S).log

[INFO] Device: cuda  SD1.5 LoRA generator
[INFO] Loading DINO...
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
[INFO] Loading SD VAE...
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/

In [ ]:
%%bash
sed -n '164,185p' /content/vsvi_project/train_vs_re_lora_gen.py
echo "=== make_schedule 반환 ==="
grep -n "def make_schedule" /content/vsvi_project/train_vs_re_latent_gen.py
sed -n '/def make_schedule/,/return/p' /content/vsvi_project/train_vs_re_latent_gen.py | head -25

def sample_sd_ddim(unet, cond_proj, eeg_lat, acp, T, steps=50, device='cuda', guidance=1.0):
    """DDIM sampling with SD 1.5 UNet conditioned on EEG."""
    B = eeg_lat.size(0)
    cond_tokens = cond_proj(eeg_lat)  # (B, N_tok, 768)

    seq = list(range(0, T, T // steps))
    x = torch.randn(B, 4, 64, 64, device=device)

    for i in reversed(range(len(seq))):
        t_val = seq[i]
        t_tensor = torch.full((B,), t_val, dtype=torch.long, device=device)
        noise_pred = unet(x, t_tensor, encoder_hidden_states=cond_tokens).sample
        a  = acp[seq[i]]
        a_ = acp[seq[i-1]] if i > 0 else torch.tensor(1.0, device=device)
        x0_pred = (x - (1-a).sqrt() * noise_pred) / a.sqrt()
        x0_pred = x0_pred.clamp(-1, 1)
        x = a_.sqrt() * x0_pred + (1 - a_).sqrt() * noise_pred
    return x


@torch.no_grad()
def evaluate_lora(unet, cond_proj, eeg_enc, test_loader, vae, dino,
=== make_schedule 반환 ===
267:def make_schedule(T, device):
def make_schedule(T, device):
    

In [ ]:
%%bash
grep -n "acp\|make_schedule" /content/vsvi_project/train_exp43_vi_lora.py | head -10

40:from train_vs_re_latent_gen import build_eeg_encoder, make_schedule
337:    acp,
423:            xt, noise = ddpm_q_sample(x0, t, acp)
481:        acp,
600:    _, _, acp = make_schedule(args.num_timesteps, device)
675:                acp,


In [ ]:
from pathlib import Path

root = Path("/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_multi")
manifests = sorted((root / "seed42" / "S09").rglob("generation_manifest.csv"))

print("S09 manifests:", len(manifests))
for p in manifests:
    print(p)

S09 manifests: 3
/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_multi/seed42/S09/full_vs_to_vi/generation_manifest.csv
/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_multi/seed42/S09/gated_residual/generation_manifest.csv
/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_multi/seed42/S09/vi_only/generation_manifest.csv


In [ ]:
import subprocess, sys, re
from pathlib import Path

SCRIPT = "/content/vsvi_project/eval_vi_rawtf_downstream_generation.py"

DATA_ROOT = "/content/drive/MyDrive/vsvi_data/preproc_vi_re"
IMG_ROOT = "/content/vsvi_project/preproc_data_vi/images"
OUT_ROOT = Path("/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_multi")

CKPTS = {
    1: {
        "vi_only": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S01/raw_tf/vi_only/encoder_best.pt",
        "full_vs_to_vi": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S01/raw_tf/vs_to_vi/encoder_best.pt",
        "gated": "/content/drive/MyDrive/vsvi_data/vi_rawtf_gated_residual_confirmatory/seed42/S01/gated_residual_vs_to_vi/encoder_best.pt",
        "generator": "/content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260617_074245_lora_r32_ep100/subj01_lora_best.pt",
    },
    2: {
        "vi_only": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S02/raw_tf/vi_only/encoder_best.pt",
        "full_vs_to_vi": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S02/raw_tf/vs_to_vi/encoder_best.pt",
        "gated": "/content/drive/MyDrive/vsvi_data/vi_rawtf_gated_residual_confirmatory/seed42/S02/gated_residual_vs_to_vi/encoder_best.pt",
        "generator": "/content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260707_133400_lora_r32_ep100/subj02_lora_best.pt",
    },
    18: {
        "vi_only": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S18/raw_tf/vi_only/encoder_best.pt",
        "full_vs_to_vi": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S18/raw_tf/vs_to_vi/encoder_best.pt",
        "gated": "/content/drive/MyDrive/vsvi_data/vi_rawtf_gated_residual_confirmatory/seed42/S18/gated_residual_vs_to_vi/encoder_best.pt",
        "generator": "/content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260612_010728_lora_r16_ep100/subj18_lora_best.pt",
    },
    28: {
        "vi_only": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S28/raw_tf/vi_only/encoder_best.pt",
        "full_vs_to_vi": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S28/raw_tf/vs_to_vi/encoder_best.pt",
        "gated": "/content/drive/MyDrive/vsvi_data/vi_rawtf_gated_residual_confirmatory/seed42/S28/gated_residual_vs_to_vi/encoder_best.pt",
        "generator": "/content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260709_191616_lora_r32_ep100/subj28_lora_best.pt",
    },
    29: {
        "vi_only": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S29/raw_tf/vi_only/encoder_best.pt",
        "full_vs_to_vi": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S29/raw_tf/vs_to_vi/encoder_best.pt",
        "gated": "/content/drive/MyDrive/vsvi_data/vi_rawtf_gated_residual_confirmatory/seed42/S29/gated_residual_vs_to_vi/encoder_best.pt",
        "generator": "/content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260710_003529_lora_r32_ep100/subj29_lora_best.pt",
    },
}

for sid, c in CKPTS.items():
    done = sorted((OUT_ROOT / "seed42" / f"S{sid:02d}").rglob("generation_manifest.csv"))
    if len(done) >= 3:
        print(f"[SKIP] S{sid:02d}: manifests={len(done)}")
        continue

    for name, path in c.items():
        assert Path(path).exists(), f"S{sid:02d} missing {name}: {path}"

    m = re.search(r"lora_r(\d+)", c["generator"])
    lora_r = int(m.group(1)) if m else 32

    print("\n" + "=" * 100)
    print(f"START S{sid:02d} lora_r={lora_r}")
    print("=" * 100)

    cmd = [
        sys.executable, "-u", SCRIPT,
        "--subject_id", str(sid),
        "--data_root", DATA_ROOT,
        "--img_root", IMG_ROOT,
        "--vi_only_ckpt", c["vi_only"],
        "--full_vs_to_vi_ckpt", c["full_vs_to_vi"],
        "--gated_ckpt", c["gated"],
        "--generator_ckpt", c["generator"],
        "--out_root", str(OUT_ROOT),
        "--seed", "42",
        "--samples_per_class", "2",
        "--generations_per_trial", "1",
        "--ddim_steps", "30",
        "--lora_r", str(lora_r),
        "--lora_alpha", str(lora_r),
        "--fp16",
    ]

    result = subprocess.run(cmd)
    print("RETURN CODE:", result.returncode)
    assert result.returncode == 0

[SKIP] S01: manifests=3
[SKIP] S02: manifests=3
[SKIP] S18: manifests=3
[SKIP] S28: manifests=3
[SKIP] S29: manifests=3


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOTS = [
    Path("/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_multi"),
    Path("/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_s24"),
]

manifests = []
for root in ROOTS:
    manifests.extend(sorted(root.rglob("generation_manifest.csv")))

print("manifests:", len(manifests))
for p in manifests:
    print(p)

rows = []

for path in manifests:
    df = pd.read_csv(path)

    subj = next((x for x in path.parts if x.startswith("S") and x[1:].isdigit()), None)
    encoder = path.parent.name

    pred_col = "pred_label" if "pred_label" in df.columns else "top1_label"
    true_col = "true_label"

    source_candidates = [
        "source_label", "conditioning_label", "condition_label",
        "eeg_source_label", "source_class", "conditioning_class",
    ]
    source_col = next((c for c in source_candidates if c in df.columns), None)

    for cond, g in df.groupby("conditioning"):
        if cond not in ["correct", "shuffled", "zero"]:
            continue

        pred = g[pred_col].to_numpy()
        true = g[true_col].to_numpy()

        original_top1 = (pred == true).mean()

        if source_col is not None:
            source = g[source_col].to_numpy()
        elif cond == "correct":
            source = true
        elif cond == "shuffled":
            source = (true + 1) % 9
        else:
            source = np.full_like(true, -1)

        source_following = (pred == source).mean() if cond != "zero" else np.nan

        counts = pd.Series(pred).value_counts().reindex(range(9), fill_value=0).to_list()
        dominant_ratio = max(counts) / sum(counts)

        rows.append({
            "subject": subj,
            "encoder": encoder,
            "conditioning": cond,
            "n": len(g),
            "original_label_top1": original_top1,
            "source_following_top1": source_following,
            "dominant_ratio": dominant_ratio,
            "prediction_counts": counts,
        })

gen = pd.DataFrame(rows)
display(gen.sort_values(["subject", "encoder", "conditioning"]))

manifests: 21
/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_multi/seed42/S01/full_vs_to_vi/generation_manifest.csv
/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_multi/seed42/S01/gated_residual/generation_manifest.csv
/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_multi/seed42/S01/vi_only/generation_manifest.csv
/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_multi/seed42/S02/full_vs_to_vi/generation_manifest.csv
/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_multi/seed42/S02/gated_residual/generation_manifest.csv
/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_multi/seed42/S02/vi_only/generation_manifest.csv
/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_multi/seed42/S09/full_vs_to_vi/generation_manifest.csv
/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_multi/seed42/S09/gated_residual/generation_manifest.csv
/content/drive/MyDrive/vsvi_data/vi_rawtf_d

,subject,encoder,conditioning,n,original_label_top1,source_following_top1,dominant_ratio,prediction_counts
0,S01,full_vs_to_vi,correct,18,0.111111,0.111111,0.388889,"[0, 0, 1, 2, 2, 3, 0, 3, 7]"
1,S01,full_vs_to_vi,shuffled,18,0.055556,0.222222,0.500000,"[0, 1, 2, 1, 3, 0, 1, 1, 9]"
2,S01,full_vs_to_vi,zero,18,0.111111,NaN,0.388889,"[0, 1, 2, 3, 7, 2, 0, 1, 2]"
3,S01,gated_residual,correct,18,0.166667,0.166667,0.388889,"[0, 0, 1, 1, 3, 4, 2, 0, 7]"
4,S01,gated_residual,shuffled,18,0.111111,0.166667,0.444444,"[0, 0, 1, 1, 1, 4, 2, 1, 8]"
...,...,...,...,...,...,...,...,...
49,S29,gated_residual,shuffled,18,0.111111,0.166667,0.222222,"[3, 0, 3, 4, 1, 3, 1, 1, 2]"
50,S29,gated_residual,zero,18,0.166667,NaN,0.333333,"[0, 0, 2, 1, 5, 2, 0, 2, 6]"
51,S29,vi_only,correct,18,0.277778,0.277778,0.277778,"[3, 0, 1, 2, 1, 5, 0, 1, 5]"
52,S29,vi_only,shuffled,18,0.111111,0.166667,0.277778,"[3, 2, 2, 2, 0, 5, 1, 1, 2]"


In [ ]:
pivot = gen[gen["conditioning"].isin(["correct", "shuffled"])].pivot_table(
    index=["subject", "encoder"],
    columns="conditioning",
    values="source_following_top1",
)

pivot["shuffled_minus_correct"] = pivot["shuffled"] - pivot["correct"]

display(pivot.reset_index().sort_values(["encoder", "subject"]))

group = (
    pivot.reset_index()
    .groupby("encoder")["shuffled_minus_correct"]
    .agg(["count", "mean", "median", "std"])
    .sort_values("mean", ascending=False)
)

display(group)

conditioning,subject,encoder,correct,shuffled,shuffled_minus_correct
0,S01,full_vs_to_vi,0.111111,0.222222,0.111111
3,S02,full_vs_to_vi,0.055556,0.111111,0.055556
6,S09,full_vs_to_vi,0.111111,0.166667,0.055556
9,S18,full_vs_to_vi,0.111111,0.000000,-0.111111
12,S24,full_vs_to_vi,0.055556,0.277778,0.222222
15,S28,full_vs_to_vi,0.111111,0.166667,0.055556
18,S29,full_vs_to_vi,0.222222,0.222222,0.000000
1,S01,gated_residual,0.166667,0.166667,0.000000
4,S02,gated_residual,0.166667,0.111111,-0.055556
7,S09,gated_residual,0.055556,0.166667,0.111111


,count,mean,median,std
encoder,,,,
full_vs_to_vi,7,0.055556,0.055556,0.101430
gated_residual,7,0.007937,0.000000,0.059391
vi_only,7,-0.015873,0.000000,0.052844


In [ ]:
orig = gen[gen["conditioning"].isin(["correct", "shuffled"])].pivot_table(
    index=["subject", "encoder"],
    columns="conditioning",
    values="original_label_top1",
)

orig["shuffled_minus_correct"] = orig["shuffled"] - orig["correct"]

display(orig.reset_index().sort_values(["encoder", "subject"]))

orig_group = (
    orig.reset_index()
    .groupby("encoder")["shuffled_minus_correct"]
    .agg(["count", "mean", "median", "std"])
    .sort_values("mean", ascending=False)
)

display(orig_group)

print(orig_group.to_string())

conditioning,subject,encoder,correct,shuffled,shuffled_minus_correct
0,S01,full_vs_to_vi,0.111111,0.055556,-0.055556
3,S02,full_vs_to_vi,0.055556,0.111111,0.055556
6,S09,full_vs_to_vi,0.111111,0.166667,0.055556
9,S18,full_vs_to_vi,0.111111,0.000000,-0.111111
12,S24,full_vs_to_vi,0.055556,0.055556,0.000000
15,S28,full_vs_to_vi,0.111111,0.111111,0.000000
18,S29,full_vs_to_vi,0.222222,0.222222,0.000000
1,S01,gated_residual,0.166667,0.111111,-0.055556
4,S02,gated_residual,0.166667,0.166667,0.000000
7,S09,gated_residual,0.055556,0.055556,0.000000


,count,mean,median,std
encoder,,,,
full_vs_to_vi,7,-0.007937,0.000000,0.059391
gated_residual,7,-0.015873,0.000000,0.069643
vi_only,7,-0.023810,-0.055556,0.089908


                count      mean    median       std
encoder                                            
full_vs_to_vi       7 -0.007937  0.000000  0.059391
gated_residual      7 -0.015873  0.000000  0.069643
vi_only             7 -0.023810 -0.055556  0.089908


In [ ]:
import subprocess, sys, re
from pathlib import Path

SCRIPT = "/content/vsvi_project/eval_vi_rawtf_downstream_generation.py"

DATA_ROOT = "/content/drive/MyDrive/vsvi_data/preproc_vi_re"
IMG_ROOT = "/content/vsvi_project/preproc_data_vi/images"
OUT_ROOT = Path("/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_spc5")

SUBJECTS = [9, 18, 24]

CKPTS_SPC5 = {
    9: {
        "vi_only": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S09/raw_tf/vi_only/encoder_best.pt",
        "full_vs_to_vi": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S09/raw_tf/vs_to_vi/encoder_best.pt",
        "gated": "/content/drive/MyDrive/vsvi_data/vi_rawtf_gated_residual_confirmatory/seed42/S09/gated_residual_vs_to_vi/encoder_best.pt",
        "generator": "/content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260710_053837_lora_r32_ep100/subj09_lora_best.pt",
    },
    18: {
        "vi_only": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S18/raw_tf/vi_only/encoder_best.pt",
        "full_vs_to_vi": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S18/raw_tf/vs_to_vi/encoder_best.pt",
        "gated": "/content/drive/MyDrive/vsvi_data/vi_rawtf_gated_residual_confirmatory/seed42/S18/gated_residual_vs_to_vi/encoder_best.pt",
        "generator": "/content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260612_010728_lora_r16_ep100/subj18_lora_best.pt",
    },
    24: {
        "vi_only": "/content/drive/MyDrive/vsvi_data/vi_tf_representation_ablation/seed42/S24/raw_tf/vi_only/encoder_best.pt",
        "full_vs_to_vi": "/content/drive/MyDrive/vsvi_data/vi_tf_representation_ablation/seed42/S24/raw_tf/vs_to_vi/encoder_best.pt",
        "gated": "/content/drive/MyDrive/vsvi_data/vi_rawtf_gated_residual_s24/seed42/S24/gated_residual_vs_to_vi/encoder_best.pt",
        "generator": "/content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260629_094904_lora_r32_ep100/subj24_lora_best.pt",
    },
}

for sid in SUBJECTS:
    c = CKPTS_SPC5[sid]

    for name, path in c.items():
        assert Path(path).exists(), f"S{sid:02d} missing {name}: {path}"

    m = re.search(r"lora_r(\d+)", c["generator"])
    lora_r = int(m.group(1)) if m else 32

    print("\n" + "=" * 100)
    print(f"START S{sid:02d} samples_per_class=5 lora_r={lora_r}")
    print("=" * 100)

    cmd = [
        sys.executable, "-u", SCRIPT,
        "--subject_id", str(sid),
        "--data_root", DATA_ROOT,
        "--img_root", IMG_ROOT,
        "--vi_only_ckpt", c["vi_only"],
        "--full_vs_to_vi_ckpt", c["full_vs_to_vi"],
        "--gated_ckpt", c["gated"],
        "--generator_ckpt", c["generator"],
        "--out_root", str(OUT_ROOT),
        "--seed", "42",
        "--samples_per_class", "5",
        "--generations_per_trial", "1",
        "--ddim_steps", "30",
        "--lora_r", str(lora_r),
        "--lora_alpha", str(lora_r),
        "--fp16",
        "--overwrite",
    ]

    result = subprocess.run(cmd)
    print("RETURN CODE:", result.returncode)
    assert result.returncode == 0


START S09 samples_per_class=5 lora_r=32
RETURN CODE: 1


AssertionError: 

In [ ]:
import glob
from pathlib import Path

patterns = [
    "/content/drive/MyDrive/vsvi_data/**/S09/**/vi_only/**/encoder_best.pt",
    "/content/drive/MyDrive/vsvi_data/**/S09/**/raw_tf/**/encoder_best.pt",
    "/content/drive/MyDrive/vsvi_data/**/S09/**/encoder_best.pt",
]

hits = []
for pat in patterns:
    hits.extend(glob.glob(pat, recursive=True))

hits = sorted(set(hits))

print("hits:", len(hits))
for p in hits:
    print(p)

hits: 12
/content/drive/MyDrive/vsvi_data/vi_ea_ablation/seed42/S09/vi_only/encoder_best.pt
/content/drive/MyDrive/vsvi_data/vi_ea_ablation/seed42/S09/vs_pretrain/encoder_best.pt
/content/drive/MyDrive/vsvi_data/vi_ea_ablation/seed42/S09/vs_to_vi/encoder_best.pt
/content/drive/MyDrive/vsvi_data/vi_encoder_transfer_multi_exp26/seed42/S09/vi_only/encoder_best.pt
/content/drive/MyDrive/vsvi_data/vi_encoder_transfer_multi_exp26/seed42/S09/vs_to_vi/encoder_best.pt
/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S09/raw/vi_only/encoder_best.pt
/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S09/raw/vs_pretrain/encoder_best.pt
/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S09/raw/vs_to_vi/encoder_best.pt
/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S09/raw_tf/vi_only/encoder_best.pt
/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S09/raw_tf/vs_pretrain/encoder_best.pt
/content/drive/MyDriv

In [ ]:
from pathlib import Path

CKPTS = {
    1: {
        "vi_only": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S01/raw_tf/vi_only/encoder_best.pt",
        "full_vs_to_vi": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S01/raw_tf/vs_to_vi/encoder_best.pt",
        "gated": "/content/drive/MyDrive/vsvi_data/vi_rawtf_gated_residual_confirmatory/seed42/S01/gated_residual_vs_to_vi/encoder_best.pt",
        "generator": "/content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260617_074245_lora_r32_ep100/subj01_lora_best.pt",
    },
    2: {
        "vi_only": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S02/raw_tf/vi_only/encoder_best.pt",
        "full_vs_to_vi": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S02/raw_tf/vs_to_vi/encoder_best.pt",
        "gated": "/content/drive/MyDrive/vsvi_data/vi_rawtf_gated_residual_confirmatory/seed42/S02/gated_residual_vs_to_vi/encoder_best.pt",
        "generator": "/content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260707_133400_lora_r32_ep100/subj02_lora_best.pt",
    },
    9: {
        "vi_only": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S09/raw_tf/vi_only/encoder_best.pt",
        "full_vs_to_vi": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S09/raw_tf/vs_to_vi/encoder_best.pt",
        "gated": "/content/drive/MyDrive/vsvi_data/vi_rawtf_gated_residual_confirmatory/seed42/S09/gated_residual_vs_to_vi/encoder_best.pt",
        "generator": "/content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260710_053837_lora_r32_ep100/subj09_lora_best.pt",
    },
    18: {
        "vi_only": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S18/raw_tf/vi_only/encoder_best.pt",
        "full_vs_to_vi": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S18/raw_tf/vs_to_vi/encoder_best.pt",
        "gated": "/content/drive/MyDrive/vsvi_data/vi_rawtf_gated_residual_confirmatory/seed42/S18/gated_residual_vs_to_vi/encoder_best.pt",
        "generator": "/content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260612_010728_lora_r16_ep100/subj18_lora_best.pt",
    },
    28: {
        "vi_only": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S28/raw_tf/vi_only/encoder_best.pt",
        "full_vs_to_vi": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S28/raw_tf/vs_to_vi/encoder_best.pt",
        "gated": "/content/drive/MyDrive/vsvi_data/vi_rawtf_gated_residual_confirmatory/seed42/S28/gated_residual_vs_to_vi/encoder_best.pt",
        "generator": "/content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260709_191616_lora_r32_ep100/subj28_lora_best.pt",
    },
    29: {
        "vi_only": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S29/raw_tf/vi_only/encoder_best.pt",
        "full_vs_to_vi": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S29/raw_tf/vs_to_vi/encoder_best.pt",
        "gated": "/content/drive/MyDrive/vsvi_data/vi_rawtf_gated_residual_confirmatory/seed42/S29/gated_residual_vs_to_vi/encoder_best.pt",
        "generator": "/content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260710_003529_lora_r32_ep100/subj29_lora_best.pt",
    },
}

In [ ]:
CKPTS[24] = {
    "vi_only": "/content/drive/MyDrive/vsvi_data/vi_tf_representation_ablation/seed42/S24/raw_tf/vi_only/encoder_best.pt",
    "full_vs_to_vi": "/content/drive/MyDrive/vsvi_data/vi_tf_representation_ablation/seed42/S24/raw_tf/vs_to_vi/encoder_best.pt",
    "gated": "/content/drive/MyDrive/vsvi_data/vi_rawtf_gated_residual_s24/seed42/S24/gated_residual_vs_to_vi/encoder_best.pt",
    "generator": "/content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260629_094904_lora_r32_ep100/subj24_lora_best.pt",
}

In [ ]:
from pathlib import Path

roots = [
    Path("/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_multi"),
    Path("/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_s24"),
    Path("/content/drive/MyDrive/vsvi_data/downstream_generation_manifests_only"),
]

for root in roots:
    manifests = sorted(root.rglob("generation_manifest.csv")) if root.exists() else []
    print("\n", root)
    print("manifests:", len(manifests))
    for p in manifests:
        print(" ", p)


 /content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_multi
manifests: 0

 /content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_s24
manifests: 0

 /content/drive/MyDrive/vsvi_data/downstream_generation_manifests_only
manifests: 21
  /content/drive/MyDrive/vsvi_data/downstream_generation_manifests_only/vi_rawtf_downstream_generation_multi/seed42/S01/full_vs_to_vi/generation_manifest.csv
  /content/drive/MyDrive/vsvi_data/downstream_generation_manifests_only/vi_rawtf_downstream_generation_multi/seed42/S01/gated_residual/generation_manifest.csv
  /content/drive/MyDrive/vsvi_data/downstream_generation_manifests_only/vi_rawtf_downstream_generation_multi/seed42/S01/vi_only/generation_manifest.csv
  /content/drive/MyDrive/vsvi_data/downstream_generation_manifests_only/vi_rawtf_downstream_generation_multi/seed42/S02/full_vs_to_vi/generation_manifest.csv
  /content/drive/MyDrive/vsvi_data/downstream_generation_manifests_only/vi_rawtf_downstream_generation_multi/seed42/S02

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/content/drive/MyDrive/vsvi_data/downstream_generation_manifests_only")
manifests = sorted(ROOT.rglob("generation_manifest.csv"))

print("manifests:", len(manifests))
assert len(manifests) == 21

rows = []

for path in manifests:
    df = pd.read_csv(path)

    subj = next((x for x in path.parts if x.startswith("S") and x[1:].isdigit()), None)
    encoder = path.parent.name

    pred_col = "pred_label" if "pred_label" in df.columns else "top1_label"
    true_col = "true_label"

    source_candidates = [
        "source_label", "conditioning_label", "condition_label",
        "eeg_source_label", "source_class", "conditioning_class",
    ]
    source_col = next((c for c in source_candidates if c in df.columns), None)

    for cond, g in df.groupby("conditioning"):
        if cond not in ["correct", "shuffled", "zero"]:
            continue

        pred = g[pred_col].to_numpy()
        true = g[true_col].to_numpy()

        original_top1 = (pred == true).mean()

        if source_col is not None:
            source = g[source_col].to_numpy()
        elif cond == "correct":
            source = true
        elif cond == "shuffled":
            source = (true + 1) % 9
        else:
            source = np.full_like(true, -1)

        source_following = (pred == source).mean() if cond != "zero" else np.nan

        counts = pd.Series(pred).value_counts().reindex(range(9), fill_value=0).to_list()
        dominant_ratio = max(counts) / sum(counts)

        rows.append({
            "subject": subj,
            "encoder": encoder,
            "conditioning": cond,
            "n": len(g),
            "original_label_top1": original_top1,
            "source_following_top1": source_following,
            "dominant_ratio": dominant_ratio,
            "prediction_counts": counts,
        })

gen = pd.DataFrame(rows)
display(gen.sort_values(["subject", "encoder", "conditioning"]))

manifests: 21


,subject,encoder,conditioning,n,original_label_top1,source_following_top1,dominant_ratio,prediction_counts
0,S01,full_vs_to_vi,correct,18,0.111111,0.111111,0.388889,"[0, 0, 1, 2, 2, 3, 0, 3, 7]"
1,S01,full_vs_to_vi,shuffled,18,0.055556,0.222222,0.500000,"[0, 1, 2, 1, 3, 0, 1, 1, 9]"
2,S01,full_vs_to_vi,zero,18,0.111111,NaN,0.388889,"[0, 1, 2, 3, 7, 2, 0, 1, 2]"
3,S01,gated_residual,correct,18,0.166667,0.166667,0.388889,"[0, 0, 1, 1, 3, 4, 2, 0, 7]"
4,S01,gated_residual,shuffled,18,0.111111,0.166667,0.444444,"[0, 0, 1, 1, 1, 4, 2, 1, 8]"
...,...,...,...,...,...,...,...,...
49,S29,gated_residual,shuffled,18,0.111111,0.166667,0.222222,"[3, 0, 3, 4, 1, 3, 1, 1, 2]"
50,S29,gated_residual,zero,18,0.166667,NaN,0.333333,"[0, 0, 2, 1, 5, 2, 0, 2, 6]"
51,S29,vi_only,correct,18,0.277778,0.277778,0.277778,"[3, 0, 1, 2, 1, 5, 0, 1, 5]"
52,S29,vi_only,shuffled,18,0.111111,0.166667,0.277778,"[3, 2, 2, 2, 0, 5, 1, 1, 2]"


In [ ]:
pivot = gen[gen["conditioning"].isin(["correct", "shuffled"])].pivot_table(
    index=["subject", "encoder"],
    columns="conditioning",
    values="source_following_top1",
)

pivot["shuffled_minus_correct"] = pivot["shuffled"] - pivot["correct"]

display(pivot.reset_index().sort_values(["encoder", "subject"]))

source_group = (
    pivot.reset_index()
    .groupby("encoder")["shuffled_minus_correct"]
    .agg(["count", "mean", "median", "std"])
    .sort_values("mean", ascending=False)
)

display(source_group)
print(source_group.to_string())

conditioning,subject,encoder,correct,shuffled,shuffled_minus_correct
0,S01,full_vs_to_vi,0.111111,0.222222,0.111111
3,S02,full_vs_to_vi,0.055556,0.111111,0.055556
6,S09,full_vs_to_vi,0.111111,0.166667,0.055556
9,S18,full_vs_to_vi,0.111111,0.000000,-0.111111
12,S24,full_vs_to_vi,0.055556,0.277778,0.222222
15,S28,full_vs_to_vi,0.111111,0.166667,0.055556
18,S29,full_vs_to_vi,0.222222,0.222222,0.000000
1,S01,gated_residual,0.166667,0.166667,0.000000
4,S02,gated_residual,0.166667,0.111111,-0.055556
7,S09,gated_residual,0.055556,0.166667,0.111111


,count,mean,median,std
encoder,,,,
full_vs_to_vi,7,0.055556,0.055556,0.101430
gated_residual,7,0.007937,0.000000,0.059391
vi_only,7,-0.015873,0.000000,0.052844


                count      mean    median       std
encoder                                            
full_vs_to_vi       7  0.055556  0.055556  0.101430
gated_residual      7  0.007937  0.000000  0.059391
vi_only             7 -0.015873  0.000000  0.052844


In [ ]:
orig = gen[gen["conditioning"].isin(["correct", "shuffled"])].pivot_table(
    index=["subject", "encoder"],
    columns="conditioning",
    values="original_label_top1",
)

orig["shuffled_minus_correct"] = orig["shuffled"] - orig["correct"]

display(orig.reset_index().sort_values(["encoder", "subject"]))

orig_group = (
    orig.reset_index()
    .groupby("encoder")["shuffled_minus_correct"]
    .agg(["count", "mean", "median", "std"])
    .sort_values("mean", ascending=False)
)

display(orig_group)
print(orig_group.to_string())

conditioning,subject,encoder,correct,shuffled,shuffled_minus_correct
0,S01,full_vs_to_vi,0.111111,0.055556,-0.055556
3,S02,full_vs_to_vi,0.055556,0.111111,0.055556
6,S09,full_vs_to_vi,0.111111,0.166667,0.055556
9,S18,full_vs_to_vi,0.111111,0.000000,-0.111111
12,S24,full_vs_to_vi,0.055556,0.055556,0.000000
15,S28,full_vs_to_vi,0.111111,0.111111,0.000000
18,S29,full_vs_to_vi,0.222222,0.222222,0.000000
1,S01,gated_residual,0.166667,0.111111,-0.055556
4,S02,gated_residual,0.166667,0.166667,0.000000
7,S09,gated_residual,0.055556,0.055556,0.000000


,count,mean,median,std
encoder,,,,
full_vs_to_vi,7,-0.007937,0.000000,0.059391
gated_residual,7,-0.015873,0.000000,0.069643
vi_only,7,-0.023810,-0.055556,0.089908


                count      mean    median       std
encoder                                            
full_vs_to_vi       7 -0.007937  0.000000  0.059391
gated_residual      7 -0.015873  0.000000  0.069643
vi_only             7 -0.023810 -0.055556  0.089908


In [ ]:
import subprocess, sys, re
from pathlib import Path

SCRIPT = "/content/vsvi_project/eval_vi_rawtf_downstream_generation.py"

DATA_ROOT = "/content/drive/MyDrive/vsvi_data/preproc_vi_re"
IMG_ROOT = "/content/vsvi_project/preproc_data_vi/images"
OUT_ROOT = Path("/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_spc5")

SUBJECTS = [9, 18, 24]

CKPTS_SPC5 = {
    9: {
        "vi_only": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S09/raw_tf/vi_only/encoder_best.pt",
        "full_vs_to_vi": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S09/raw_tf/vs_to_vi/encoder_best.pt",
        "gated": "/content/drive/MyDrive/vsvi_data/vi_rawtf_gated_residual_confirmatory/seed42/S09/gated_residual_vs_to_vi/encoder_best.pt",
        "generator": "/content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260710_053837_lora_r32_ep100/subj09_lora_best.pt",
    },
    18: {
        "vi_only": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S18/raw_tf/vi_only/encoder_best.pt",
        "full_vs_to_vi": "/content/drive/MyDrive/vsvi_data/vi_rawtf_confirmatory_multi/seed42/S18/raw_tf/vs_to_vi/encoder_best.pt",
        "gated": "/content/drive/MyDrive/vsvi_data/vi_rawtf_gated_residual_confirmatory/seed42/S18/gated_residual_vs_to_vi/encoder_best.pt",
        "generator": "/content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260612_010728_lora_r16_ep100/subj18_lora_best.pt",
    },
    24: {
        "vi_only": "/content/drive/MyDrive/vsvi_data/vi_tf_representation_ablation/seed42/S24/raw_tf/vi_only/encoder_best.pt",
        "full_vs_to_vi": "/content/drive/MyDrive/vsvi_data/vi_tf_representation_ablation/seed42/S24/raw_tf/vs_to_vi/encoder_best.pt",
        "gated": "/content/drive/MyDrive/vsvi_data/vi_rawtf_gated_residual_s24/seed42/S24/gated_residual_vs_to_vi/encoder_best.pt",
        "generator": "/content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260629_094904_lora_r32_ep100/subj24_lora_best.pt",
    },
}

for sid in SUBJECTS:
    c = CKPTS_SPC5[sid]

    for name, path in c.items():
        assert Path(path).exists(), f"S{sid:02d} missing {name}: {path}"

    m = re.search(r"lora_r(\d+)", c["generator"])
    lora_r = int(m.group(1)) if m else 32

    print("\n" + "=" * 100)
    print(f"START S{sid:02d} samples_per_class=5 lora_r={lora_r}")
    print("=" * 100)

    cmd = [
        sys.executable, "-u", SCRIPT,
        "--subject_id", str(sid),
        "--data_root", DATA_ROOT,
        "--img_root", IMG_ROOT,
        "--vi_only_ckpt", c["vi_only"],
        "--full_vs_to_vi_ckpt", c["full_vs_to_vi"],
        "--gated_ckpt", c["gated"],
        "--generator_ckpt", c["generator"],
        "--out_root", str(OUT_ROOT),
        "--seed", "42",
        "--samples_per_class", "5",
        "--generations_per_trial", "1",
        "--ddim_steps", "30",
        "--lora_r", str(lora_r),
        "--lora_alpha", str(lora_r),
        "--fp16",
        "--overwrite",
    ]

    result = subprocess.run(cmd)
    print("RETURN CODE:", result.returncode)
    assert result.returncode == 0

In [ ]:
import subprocess, sys, re
from pathlib import Path

SCRIPT = "/content/vsvi_project/eval_vi_rawtf_downstream_generation.py"

sid = 9
c = CKPTS_SPC5[sid]

m = re.search(r"lora_r(\d+)", c["generator"])
lora_r = int(m.group(1)) if m else 32

cmd = [
    sys.executable, "-u", SCRIPT,
    "--subject_id", str(sid),
    "--data_root", "/content/drive/MyDrive/vsvi_data/preproc_vi_re",
    "--img_root", "/content/vsvi_project/preproc_data_vi/images",
    "--vi_only_ckpt", c["vi_only"],
    "--full_vs_to_vi_ckpt", c["full_vs_to_vi"],
    "--gated_ckpt", c["gated"],
    "--generator_ckpt", c["generator"],
    "--out_root", "/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_spc5",
    "--seed", "42",
    "--samples_per_class", "5",
    "--generations_per_trial", "1",
    "--ddim_steps", "30",
    "--lora_r", str(lora_r),
    "--lora_alpha", str(lora_r),
    "--fp16",
    "--overwrite",
]

result = subprocess.run(cmd, text=True, capture_output=True)

print("RETURN CODE:", result.returncode)
print("\nSTDOUT LAST 5000\n", result.stdout[-5000:])
print("\nSTDERR LAST 5000\n", result.stderr[-5000:])

In [ ]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_spc5")
manifests = sorted(ROOT.rglob("generation_manifest.csv"))

print("manifests:", len(manifests))
for p in manifests:
    print(p)

assert len(manifests) == 9

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'uninstall', '-y', 'torchao'], returncode=0)

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_spc5")
manifests = sorted(ROOT.rglob("generation_manifest.csv"))

rows = []

for path in manifests:
    df = pd.read_csv(path)

    subj = next((x for x in path.parts if x.startswith("S") and x[1:].isdigit()), None)
    encoder = path.parent.name

    pred_col = "pred_label" if "pred_label" in df.columns else "top1_label"
    true_col = "true_label"

    source_candidates = [
        "source_label", "conditioning_label", "condition_label",
        "eeg_source_label", "source_class", "conditioning_class",
    ]
    source_col = next((c for c in source_candidates if c in df.columns), None)

    for cond, g in df.groupby("conditioning"):
        if cond not in ["correct", "shuffled", "zero"]:
            continue

        pred = g[pred_col].to_numpy()
        true = g[true_col].to_numpy()

        original_top1 = (pred == true).mean()

        if source_col is not None:
            source = g[source_col].to_numpy()
        elif cond == "correct":
            source = true
        elif cond == "shuffled":
            source = (true + 1) % 9
        else:
            source = np.full_like(true, -1)

        source_following = (pred == source).mean() if cond != "zero" else np.nan

        counts = pd.Series(pred).value_counts().reindex(range(9), fill_value=0).to_list()
        dominant_ratio = max(counts) / sum(counts)

        rows.append({
            "subject": subj,
            "encoder": encoder,
            "conditioning": cond,
            "n": len(g),
            "original_label_top1": original_top1,
            "source_following_top1": source_following,
            "dominant_ratio": dominant_ratio,
            "prediction_counts": counts,
        })

gen5 = pd.DataFrame(rows)
display(gen5.sort_values(["subject", "encoder", "conditioning"]))

In [ ]:
pivot5 = gen5[gen5["conditioning"].isin(["correct", "shuffled"])].pivot_table(
    index=["subject", "encoder"],
    columns="conditioning",
    values="source_following_top1",
)

pivot5["shuffled_minus_correct"] = pivot5["shuffled"] - pivot5["correct"]

display(pivot5.reset_index().sort_values(["encoder", "subject"]))

source_group5 = (
    pivot5.reset_index()
    .groupby("encoder")["shuffled_minus_correct"]
    .agg(["count", "mean", "median", "std"])
    .sort_values("mean", ascending=False)
)

display(source_group5)
print(source_group5.to_string())

In [ ]:
orig5 = gen5[gen5["conditioning"].isin(["correct", "shuffled"])].pivot_table(
    index=["subject", "encoder"],
    columns="conditioning",
    values="original_label_top1",
)

orig5["shuffled_minus_correct"] = orig5["shuffled"] - orig5["correct"]

display(orig5.reset_index().sort_values(["encoder", "subject"]))

orig_group5 = (
    orig5.reset_index()
    .groupby("encoder")["shuffled_minus_correct"]
    .agg(["count", "mean", "median", "std"])
    .sort_values("mean", ascending=False)
)

display(orig_group5)
print(orig_group5.to_string())

In [3]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_spc5")

print("exists:", ROOT.exists())
print("path:", ROOT)

exists: True
path: /content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_spc5


In [4]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_spc5")

manifests = sorted(ROOT.rglob("generation_manifest.csv"))
metrics = sorted(ROOT.rglob("metrics.json"))
images = sorted(ROOT.rglob("*.png"))

print("manifests:", len(manifests))
for p in manifests:
    print(" ", p)

print("\nmetrics:", len(metrics))
for p in metrics:
    print(" ", p)

print("\nimages:", len(images))
print("first images:")
for p in images[:10]:
    print(" ", p)

manifests: 9
  /content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_spc5/seed42/S09/full_vs_to_vi/generation_manifest.csv
  /content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_spc5/seed42/S09/gated_residual/generation_manifest.csv
  /content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_spc5/seed42/S09/vi_only/generation_manifest.csv
  /content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_spc5/seed42/S18/full_vs_to_vi/generation_manifest.csv
  /content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_spc5/seed42/S18/gated_residual/generation_manifest.csv
  /content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_spc5/seed42/S18/vi_only/generation_manifest.csv
  /content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_spc5/seed42/S24/full_vs_to_vi/generation_manifest.csv
  /content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_spc5/seed42/S24/gated_residual/generation_manifest.csv
  /content/drive/MyDrive/vsvi_data/v

In [5]:
from pathlib import Path
import pandas as pd

ROOT = Path("/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_spc5")

rows = []

for sid in [9, 18, 24]:
    sdir = ROOT / "seed42" / f"S{sid:02d}"
    for enc in ["vi_only", "full_vs_to_vi", "gated_residual"]:
        manifest = sdir / enc / "generation_manifest.csv"
        metrics = sdir / enc / "metrics.json"
        img_dir = sdir / enc / "images"
        rows.append({
            "subject": f"S{sid:02d}",
            "encoder": enc,
            "manifest": manifest.exists(),
            "metrics": metrics.exists(),
            "n_images": len(list(img_dir.glob("*.png"))) if img_dir.exists() else 0,
            "path": str(sdir / enc),
        })

check = pd.DataFrame(rows)
display(check)

,subject,encoder,manifest,metrics,n_images,path
0,S09,vi_only,True,False,135,/content/drive/MyDrive/vsvi_data/vi_rawtf_down...
1,S09,full_vs_to_vi,True,False,135,/content/drive/MyDrive/vsvi_data/vi_rawtf_down...
2,S09,gated_residual,True,False,135,/content/drive/MyDrive/vsvi_data/vi_rawtf_down...
3,S18,vi_only,True,False,135,/content/drive/MyDrive/vsvi_data/vi_rawtf_down...
4,S18,full_vs_to_vi,True,False,135,/content/drive/MyDrive/vsvi_data/vi_rawtf_down...
5,S18,gated_residual,True,False,135,/content/drive/MyDrive/vsvi_data/vi_rawtf_down...
6,S24,vi_only,True,False,135,/content/drive/MyDrive/vsvi_data/vi_rawtf_down...
7,S24,full_vs_to_vi,True,False,135,/content/drive/MyDrive/vsvi_data/vi_rawtf_down...
8,S24,gated_residual,True,False,135,/content/drive/MyDrive/vsvi_data/vi_rawtf_down...


In [6]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/content/drive/MyDrive/vsvi_data/vi_rawtf_downstream_generation_spc5")
manifests = sorted(ROOT.rglob("generation_manifest.csv"))

print("manifests:", len(manifests))
assert len(manifests) == 9

rows = []

for path in manifests:
    df = pd.read_csv(path)

    subj = next((x for x in path.parts if x.startswith("S") and x[1:].isdigit()), None)
    encoder = path.parent.name

    pred_col = "pred_label" if "pred_label" in df.columns else "top1_label"
    true_col = "true_label"

    source_candidates = [
        "source_label", "conditioning_label", "condition_label",
        "eeg_source_label", "source_class", "conditioning_class",
    ]
    source_col = next((c for c in source_candidates if c in df.columns), None)

    for cond, g in df.groupby("conditioning"):
        if cond not in ["correct", "shuffled", "zero"]:
            continue

        pred = g[pred_col].to_numpy()
        true = g[true_col].to_numpy()

        original_top1 = (pred == true).mean()

        if source_col is not None:
            source = g[source_col].to_numpy()
        elif cond == "correct":
            source = true
        elif cond == "shuffled":
            source = (true + 1) % 9
        else:
            source = np.full_like(true, -1)

        source_following = (pred == source).mean() if cond != "zero" else np.nan

        counts = pd.Series(pred).value_counts().reindex(range(9), fill_value=0).to_list()
        dominant_ratio = max(counts) / sum(counts)

        rows.append({
            "subject": subj,
            "encoder": encoder,
            "conditioning": cond,
            "n": len(g),
            "original_label_top1": original_top1,
            "source_following_top1": source_following,
            "dominant_ratio": dominant_ratio,
            "prediction_counts": counts,
        })

gen5 = pd.DataFrame(rows)
display(gen5.sort_values(["subject", "encoder", "conditioning"]))

manifests: 9


,subject,encoder,conditioning,n,original_label_top1,source_following_top1,dominant_ratio,prediction_counts
0,S09,full_vs_to_vi,correct,45,0.244444,0.244444,0.333333,"[2, 3, 15, 4, 8, 0, 0, 5, 8]"
1,S09,full_vs_to_vi,shuffled,45,0.066667,0.222222,0.422222,"[4, 4, 19, 1, 3, 3, 0, 6, 5]"
2,S09,full_vs_to_vi,zero,45,0.111111,NaN,0.266667,"[0, 0, 2, 5, 7, 8, 1, 12, 10]"
3,S09,gated_residual,correct,45,0.066667,0.066667,0.511111,"[8, 3, 23, 1, 6, 0, 1, 1, 2]"
4,S09,gated_residual,shuffled,45,0.022222,0.088889,0.444444,"[9, 3, 20, 0, 2, 0, 0, 1, 10]"
5,S09,gated_residual,zero,45,0.111111,NaN,0.266667,"[0, 0, 2, 5, 7, 8, 1, 12, 10]"
6,S09,vi_only,correct,45,0.088889,0.088889,0.555556,"[7, 2, 25, 0, 5, 1, 0, 0, 5]"
7,S09,vi_only,shuffled,45,0.022222,0.111111,0.533333,"[6, 1, 24, 2, 7, 0, 1, 0, 4]"
8,S09,vi_only,zero,45,0.111111,NaN,0.266667,"[0, 0, 2, 5, 7, 8, 1, 12, 10]"
9,S18,full_vs_to_vi,correct,45,0.155556,0.155556,0.288889,"[3, 0, 3, 5, 5, 5, 2, 9, 13]"


In [7]:
pivot5 = gen5[gen5["conditioning"].isin(["correct", "shuffled"])].pivot_table(
    index=["subject", "encoder"],
    columns="conditioning",
    values="source_following_top1",
)

pivot5["shuffled_minus_correct"] = pivot5["shuffled"] - pivot5["correct"]

display(pivot5.reset_index().sort_values(["encoder", "subject"]))

source_group5 = (
    pivot5.reset_index()
    .groupby("encoder")["shuffled_minus_correct"]
    .agg(["count", "mean", "median", "std"])
    .sort_values("mean", ascending=False)
)

display(source_group5)
print(source_group5.to_string())

conditioning,subject,encoder,correct,shuffled,shuffled_minus_correct
0,S09,full_vs_to_vi,0.244444,0.222222,-0.022222
3,S18,full_vs_to_vi,0.155556,0.111111,-0.044444
6,S24,full_vs_to_vi,0.088889,0.066667,-0.022222
1,S09,gated_residual,0.066667,0.088889,0.022222
4,S18,gated_residual,0.177778,0.133333,-0.044444
7,S24,gated_residual,0.044444,0.044444,0.000000
2,S09,vi_only,0.088889,0.111111,0.022222
5,S18,vi_only,0.088889,0.177778,0.088889
8,S24,vi_only,0.088889,0.066667,-0.022222


,count,mean,median,std
encoder,,,,
vi_only,3,0.029630,0.022222,0.055925
gated_residual,3,-0.007407,0.000000,0.033945
full_vs_to_vi,3,-0.029630,-0.022222,0.012830


                count      mean    median       std
encoder                                            
vi_only             3  0.029630  0.022222  0.055925
gated_residual      3 -0.007407  0.000000  0.033945
full_vs_to_vi       3 -0.029630 -0.022222  0.012830


In [8]:
orig5 = gen5[gen5["conditioning"].isin(["correct", "shuffled"])].pivot_table(
    index=["subject", "encoder"],
    columns="conditioning",
    values="original_label_top1",
)

orig5["shuffled_minus_correct"] = orig5["shuffled"] - orig5["correct"]

display(orig5.reset_index().sort_values(["encoder", "subject"]))

orig_group5 = (
    orig5.reset_index()
    .groupby("encoder")["shuffled_minus_correct"]
    .agg(["count", "mean", "median", "std"])
    .sort_values("mean", ascending=False)
)

display(orig_group5)
print(orig_group5.to_string())

conditioning,subject,encoder,correct,shuffled,shuffled_minus_correct
0,S09,full_vs_to_vi,0.244444,0.066667,-0.177778
3,S18,full_vs_to_vi,0.155556,0.066667,-0.088889
6,S24,full_vs_to_vi,0.088889,0.177778,0.088889
1,S09,gated_residual,0.066667,0.022222,-0.044444
4,S18,gated_residual,0.177778,0.088889,-0.088889
7,S24,gated_residual,0.044444,0.177778,0.133333
2,S09,vi_only,0.088889,0.022222,-0.066667
5,S18,vi_only,0.088889,0.066667,-0.022222
8,S24,vi_only,0.088889,0.155556,0.066667


,count,mean,median,std
encoder,,,,
gated_residual,3,0.000000,-0.044444,0.117589
vi_only,3,-0.007407,-0.022222,0.067890
full_vs_to_vi,3,-0.059259,-0.088889,0.135780


                count      mean    median       std
encoder                                            
gated_residual      3  0.000000 -0.044444  0.117589
vi_only             3 -0.007407 -0.022222  0.067890
full_vs_to_vi       3 -0.059259 -0.088889  0.135780


In [9]:
dom5 = gen5[gen5["conditioning"].isin(["correct", "shuffled"])].pivot_table(
    index=["subject", "encoder"],
    columns="conditioning",
    values="dominant_ratio",
)

dom5["shuffled_minus_correct"] = dom5["shuffled"] - dom5["correct"]

display(dom5.reset_index().sort_values(["encoder", "subject"]))

dom_group5 = (
    dom5.reset_index()
    .groupby("encoder")["shuffled_minus_correct"]
    .agg(["count", "mean", "median", "std"])
    .sort_values("mean", ascending=False)
)

display(dom_group5)
print(dom_group5.to_string())

conditioning,subject,encoder,correct,shuffled,shuffled_minus_correct
0,S09,full_vs_to_vi,0.333333,0.422222,0.088889
3,S18,full_vs_to_vi,0.288889,0.400000,0.111111
6,S24,full_vs_to_vi,0.444444,0.355556,-0.088889
1,S09,gated_residual,0.511111,0.444444,-0.066667
4,S18,gated_residual,0.288889,0.288889,0.000000
7,S24,gated_residual,0.488889,0.466667,-0.022222
2,S09,vi_only,0.555556,0.533333,-0.022222
5,S18,vi_only,0.444444,0.377778,-0.066667
8,S24,vi_only,0.466667,0.400000,-0.066667


,count,mean,median,std
encoder,,,,
full_vs_to_vi,3,0.037037,0.088889,0.109620
gated_residual,3,-0.029630,-0.022222,0.033945
vi_only,3,-0.051852,-0.066667,0.025660


                count      mean    median       std
encoder                                            
full_vs_to_vi       3  0.037037  0.088889  0.109620
gated_residual      3 -0.029630 -0.022222  0.033945
vi_only             3 -0.051852 -0.066667  0.025660


In [10]:
import pandas as pd
from pathlib import Path

rows = [
    {
        "setting": "spc2_all_subjects",
        "n_subjects": 7,
        "encoder": "full_vs_to_vi",
        "source_following_delta": 0.055556,
        "original_label_delta": -0.007937,
        "dominant_delta": None,
        "interpretation": "weak positive source-following, unstable",
    },
    {
        "setting": "spc2_all_subjects",
        "n_subjects": 7,
        "encoder": "gated_residual",
        "source_following_delta": 0.007937,
        "original_label_delta": -0.015873,
        "dominant_delta": None,
        "interpretation": "near zero source-following",
    },
    {
        "setting": "spc2_all_subjects",
        "n_subjects": 7,
        "encoder": "vi_only",
        "source_following_delta": -0.015873,
        "original_label_delta": -0.023810,
        "dominant_delta": None,
        "interpretation": "no source-following",
    },
    {
        "setting": "spc5_selected_subjects",
        "n_subjects": 3,
        "encoder": "full_vs_to_vi",
        "source_following_delta": -0.029630,
        "original_label_delta": -0.059259,
        "dominant_delta": 0.037037,
        "interpretation": "conditioning changes output but not source-semantically",
    },
    {
        "setting": "spc5_selected_subjects",
        "n_subjects": 3,
        "encoder": "gated_residual",
        "source_following_delta": -0.007407,
        "original_label_delta": 0.000000,
        "dominant_delta": -0.029630,
        "interpretation": "weak generation conditioning",
    },
    {
        "setting": "spc5_selected_subjects",
        "n_subjects": 3,
        "encoder": "vi_only",
        "source_following_delta": 0.029630,
        "original_label_delta": -0.007407,
        "dominant_delta": -0.051852,
        "interpretation": "small inconsistent source-following",
    },
]

df = pd.DataFrame(rows)

OUT = Path("/content/drive/MyDrive/vsvi_data/downstream_generation_transfer_summary.csv")
df.to_csv(OUT, index=False)
display(df)
print("saved:", OUT)

,setting,n_subjects,encoder,source_following_delta,original_label_delta,dominant_delta,interpretation
0,spc2_all_subjects,7,full_vs_to_vi,0.055556,-0.007937,NaN,"weak positive source-following, unstable"
1,spc2_all_subjects,7,gated_residual,0.007937,-0.015873,NaN,near zero source-following
2,spc2_all_subjects,7,vi_only,-0.015873,-0.023810,NaN,no source-following
3,spc5_selected_subjects,3,full_vs_to_vi,-0.029630,-0.059259,0.037037,conditioning changes output but not source-sem...
4,spc5_selected_subjects,3,gated_residual,-0.007407,0.000000,-0.029630,weak generation conditioning
5,spc5_selected_subjects,3,vi_only,0.029630,-0.007407,-0.051852,small inconsistent source-following


saved: /content/drive/MyDrive/vsvi_data/downstream_generation_transfer_summary.csv


In [ ]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/vsvi_data")

keywords = [
    "vi_rawtf_confirmatory_multi",
    "vi_rawtf_gated_residual_confirmatory",
    "vi_encoder_transfer_multi_exp26",
    "vi_tf_representation_ablation",
]

files = []

for p in ROOT.rglob("*"):
    if p.is_file() and p.suffix in [".csv", ".json"]:
        s = str(p)
        if any(k in s for k in keywords):
            files.append(p)

print("files:", len(files))
for p in sorted(files):
    print(p)

In [ ]:
import json
from pathlib import Path
import pandas as pd
import re

ROOT = Path("/content/drive/MyDrive/vsvi_data")

SUBJECTS = [1, 2, 9, 18, 24, 28, 29]

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def flatten(d, prefix=""):
    out = {}
    if isinstance(d, dict):
        for k, v in d.items():
            out.update(flatten(v, f"{prefix}.{k}" if prefix else str(k)))
    else:
        out[prefix] = d
    return out

rows = []

# JSON metrics 우선 수집
for p in ROOT.rglob("metrics.json"):
    s = str(p)

    if not any(k in s for k in [
        "vi_rawtf_confirmatory_multi",
        "vi_rawtf_gated_residual_confirmatory",
        "vi_tf_representation_ablation",
        "vi_encoder_transfer_multi_exp26",
    ]):
        continue

    subj_match = re.search(r"/S(\d{2})/", s)
    if not subj_match:
        continue

    sid = int(subj_match.group(1))
    if sid not in SUBJECTS:
        continue

    try:
        m = flatten(load_json(p))
    except Exception as e:
        print("failed:", p, e)
        continue

    if "/raw_tf/vi_only/" in s:
        method = "raw_tf_vi_only"
    elif "/raw_tf/vs_to_vi/" in s:
        method = "raw_tf_vs_to_vi"
    elif "/raw/vi_only/" in s:
        method = "raw_vi_only"
    elif "/raw/vs_to_vi/" in s:
        method = "raw_vs_to_vi"
    elif "gated_residual_vs_to_vi" in s:
        method = "gated_residual_vs_to_vi"
    elif "/vi_only/" in s:
        method = "vi_only_other"
    elif "/vs_to_vi/" in s:
        method = "vs_to_vi_other"
    else:
        method = "unknown"

    row = {
        "subject": sid,
        "method": method,
        "path": str(p),
    }

    for key in [
        "balanced_accuracy",
        "test_BAC",
        "VI_test_BAC",
        "target_VI_BAC",
        "top1",
        "VI_test_top1",
        "target_VI_top1",
        "top3",
        "VI_test_top3",
        "target_VI_top3",
        "top5",
        "VI_test_top5",
        "target_VI_top5",
        "dominant_ratio",
        "normalized_entropy",
        "best_epoch",
    ]:
        if key in m:
            row[key] = m[key]

    rows.append(row)

metrics = pd.DataFrame(rows)
display(metrics.sort_values(["subject", "method"]))
print("rows:", len(metrics))

In [ ]:
import numpy as np
import pandas as pd

def first_existing(row, cols):
    for c in cols:
        if c in row and pd.notna(row[c]):
            return row[c]
    return np.nan

norm_rows = []

for _, r in metrics.iterrows():
    norm_rows.append({
        "subject": int(r["subject"]),
        "method": r["method"],
        "BAC": first_existing(r, ["balanced_accuracy", "test_BAC", "VI_test_BAC", "target_VI_BAC"]),
        "Top1": first_existing(r, ["top1", "VI_test_top1", "target_VI_top1"]),
        "Top3": first_existing(r, ["top3", "VI_test_top3", "target_VI_top3"]),
        "Top5": first_existing(r, ["top5", "VI_test_top5", "target_VI_top5"]),
        "dominant_ratio": first_existing(r, ["dominant_ratio"]),
        "entropy": first_existing(r, ["normalized_entropy"]),
        "best_epoch": first_existing(r, ["best_epoch"]),
        "path": r["path"],
    })

enc = pd.DataFrame(norm_rows)
display(enc.sort_values(["subject", "method"]))

In [ ]:
summary = (
    enc.groupby("method")
    .agg(
        n=("subject", "nunique"),
        BAC_mean=("BAC", "mean"),
        BAC_median=("BAC", "median"),
        BAC_std=("BAC", "std"),
        Top3_mean=("Top3", "mean"),
        Top5_mean=("Top5", "mean"),
        dominant_mean=("dominant_ratio", "mean"),
        entropy_mean=("entropy", "mean"),
    )
    .sort_values("BAC_mean", ascending=False)
)

display(summary)

In [ ]:
wide = enc.pivot_table(
    index="subject",
    columns="method",
    values="BAC",
    aggfunc="first",
)

display(wide)

comparisons = pd.DataFrame({
    "subject": wide.index,
    "raw_tf_vs_to_vi_minus_raw_tf_vi_only": (
        wide.get("raw_tf_vs_to_vi") - wide.get("raw_tf_vi_only")
    ),
    "raw_tf_vi_only_minus_raw_vi_only": (
        wide.get("raw_tf_vi_only") - wide.get("raw_vi_only")
    ),
    "raw_tf_vs_to_vi_minus_raw_vs_to_vi": (
        wide.get("raw_tf_vs_to_vi") - wide.get("raw_vs_to_vi")
    ),
    "gated_minus_raw_tf_vi_only": (
        wide.get("gated_residual_vs_to_vi") - wide.get("raw_tf_vi_only")
    ),
    "gated_minus_raw_tf_vs_to_vi": (
        wide.get("gated_residual_vs_to_vi") - wide.get("raw_tf_vs_to_vi")
    ),
})

display(comparisons)

delta_summary = comparisons.drop(columns=["subject"]).agg(["count", "mean", "median", "std"]).T
display(delta_summary)

In [ ]:
from pathlib import Path

OUT_DIR = Path("/content/drive/MyDrive/vsvi_data/final_vs_to_vi_analysis")
OUT_DIR.mkdir(parents=True, exist_ok=True)

enc.to_csv(OUT_DIR / "encoder_subject_level_metrics.csv", index=False)
summary.to_csv(OUT_DIR / "encoder_method_summary.csv")
comparisons.to_csv(OUT_DIR / "encoder_transfer_deltas.csv", index=False)
delta_summary.to_csv(OUT_DIR / "encoder_transfer_delta_summary.csv")

print("saved to:", OUT_DIR)